In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver07/data-project/sc_val.jsonl
/kaggle/input/datasets/achiver07/data-project/acts_val.jsonl
/kaggle/input/datasets/achiver07/data-project/acts_train.jsonl
/kaggle/input/datasets/achiver07/data-project/tribunals_val.jsonl
/kaggle/input/datasets/achiver07/data-project/test.json
/kaggle/input/datasets/achiver07/data-project/sc_train.jsonl
/kaggle/input/datasets/achiver07/data-project/tribunals_train.jsonl
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
/kaggle/input/datasets/achiver07/phase1-token

In [2]:
# ============================================================
# GENERATION + SUBMISSION PACKAGING - ARCHITECTURE 4 (hybrid GQA decoder), self-contained
#
# The model code in section 0 is copied unchanged from your train_arch4_t4_v4.py (model section only), so no other cell is needed.
# Inputs: phase1-tokens-v40k-clean1 (tokenizer.json, test.json) and the Arch 4 checkpoint.
# Supports both the new plain best_model.pt + best_meta.json format and the older combined best.pt/latest.pt format.
#
# MODE = "test": official test prompts -> part file(s) -> prediction.json + submission.zip once ALL 1000 IDs exist across part files.
# MODE = "val" : held-out validation pairs, to compare decoding settings locally (needs the raw val .jsonl files).
#
# HOW TO RUN - ONE COMMAND DOES EVERYTHING:
#   1. save this file (in a notebook: first line of a cell "%%writefile gen4.py", paste this file below it, run the cell)
#   2. run:   !python gen4.py
#      It splits the test prompts over all visible GPUs (one worker process per GPU, logs gen_0.log, gen_1.log), prints progress every
#      minute, writes submission.zip, and verifies it ("READY TO UPLOAD" or "CHECK FAILED"). Finished prompts are saved every 25
#      prompts, so if anything stops, run the same command again and it continues where it left off.
#   Optional environment variables:  N_GPUS=1 (force one GPU) | CKPT=<path to best.pt or latest.pt>
#   Advanced (one worker by hand):   GPU_INDEX=1 GEN_START=0 GEN_END=5 python gen4.py     (5-prompt timing test on GPU 1)
#
# Generation uses no KV cache: each new token re-runs the prefix. The model needs a length that is a multiple of W (=256), so the
# prefix is RIGHT-padded with token 0; the model is causal, so padding after the last real token cannot change that token's logits.
# Only the last real position goes through the output head. All of this is verified numerically at start-up (section 2).
# ============================================================
import os, sys, subprocess, time


def _launch_all():
    """Default entry point (no GEN_END in the environment): split the test prompts over all visible GPUs, run one worker process per GPU
    (this same file, with GPU_INDEX / GEN_START / GEN_END set), report progress, then verify the zip."""
    import json, zipfile
    me = None
    try:
        me = os.path.abspath(__file__)
    except NameError:                                           # pasted straight into a notebook cell: save this cell's own text as gen4.py
        try:
            cell_text = get_ipython().user_ns["_ih"][-1]
            me = os.path.abspath("gen4.py")
            with open(me, "w") as f:
                f.write(cell_text)
            print(f"saved this cell as {me}", flush=True)
        except Exception:
            me = None
    if me is None:
        print('Could not save the script automatically. Put "%%writefile gen4.py" on the first line of the cell, run it, then run:  !python gen4.py')
        return 1
    tokd = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1"
    out = "/kaggle/working/submission_arch4_best"
    n_prompts = len(json.load(open(f"{tokd}/test.json")))
    try:
        listing = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines()
        n_gpu = sum(1 for line in listing if line.startswith("GPU"))
    except Exception:
        n_gpu = 0
    n_gpu = max(1, int(os.environ.get("N_GPUS", n_gpu)))
    cuts = [round(i * n_prompts / n_gpu) for i in range(n_gpu + 1)]
    jobs = [(i, cuts[i], cuts[i + 1]) for i in range(n_gpu)]
    print(f"{n_prompts} prompts over {n_gpu} GPU(s): " + ", ".join(f"GPU{i}: {a}-{b}" for i, a, b in jobs), flush=True)

    procs = []
    for i, a, b in jobs:
        env = {**os.environ, "GPU_INDEX": str(i), "GEN_START": str(a), "GEN_END": str(b), "PYTHONUNBUFFERED": "1"}
        procs.append(subprocess.Popen([sys.executable, me], env=env, stdout=open(f"gen_{i}.log", "w"), stderr=subprocess.STDOUT))

    def last_line(path):
        try:
            with open(path, "rb") as f:
                f.seek(0, 2)
                f.seek(max(0, f.tell() - 3000))
                lines = [ln for ln in f.read().decode("utf-8", "replace").splitlines() if ln.strip()]
            return lines[-1] if lines else "(starting)"
        except OSError:
            return "(no log yet)"

    t0, tick = time.time(), 0
    while any(p.poll() is None for p in procs):
        time.sleep(5)
        tick += 1
        if tick % 12 == 0:                                      # every minute
            print(f"[{(time.time() - t0) / 60:5.1f} min] " + " | ".join(f"GPU{i}: {last_line(f'gen_{i}.log')[:75]}" for i, _, _ in jobs), flush=True)

    bad = [(i, p.returncode) for (i, _, _), p in zip(jobs, procs) if p.returncode != 0]
    if bad:
        for i, rc in bad:
            print(f"\n=== GPU{i} worker FAILED (exit code {rc}); last lines of gen_{i}.log ===")
            print("\n".join(open(f"gen_{i}.log", errors="replace").read().splitlines()[-25:]))
        print("\nFinished prompts are saved. Send me the lines above; after the fix, run the same command again and it continues.")
        sys.exit(1)

    zpath = f"{out}/submission.zip"
    if not os.path.exists(zpath):                               # both workers ended at the same moment: run one merge-only pass (no prompts)
        print("zip not written yet: merging the part files ...", flush=True)
        subprocess.run([sys.executable, me], env={**os.environ, "GPU_INDEX": "0", "GEN_START": "0", "GEN_END": "0"})
    z = zipfile.ZipFile(zpath)
    pred = json.load(open(f"{out}/prediction.json", encoding="utf-8"))
    words = {len(v["text"].split()) for v in pred.values()}
    print("zip contents:", z.namelist(), "| IDs:", len(pred), "| words per text:", words)
    ok = z.namelist() == ["prediction.json"] and len(pred) == n_prompts and words == {128}
    print(f"READY TO UPLOAD: {zpath}" if ok else "CHECK FAILED: do not upload; send me this output")
    sys.exit(0 if ok else 1)


if "GEN_END" not in os.environ:
    _launch_all()

import json, math, random, zipfile, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

# ============================================================
# 0. MODEL DEFINITION (copied unchanged from train_arch4_t4_v4.py)
# ============================================================
def rotate_half(x):
    h = x.shape[-1] // 2
    return torch.cat([-x[..., h:], x[..., :h]], dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base):
        super().__init__()
        inv = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        f = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv)
        emb = torch.cat([f, f], dim=-1)
        self.register_buffer("cos", emb.cos(), persistent=False)
        self.register_buffer("sin", emb.sin(), persistent=False)

    def forward(self, T):
        return self.cos[:T][None, None], self.sin[:T][None, None]


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.in_proj = nn.Linear(d_model, 2 * d_ff, bias=False)
        self.out_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        g, u = self.in_proj(x).chunk(2, dim=-1)
        return self.out_proj(F.silu(g) * u)


class GQAAttention(nn.Module):
    def __init__(self, d_model, n_q, n_kv, window, is_global, dropout=0.0):
        super().__init__()
        assert d_model % n_q == 0 and n_q % n_kv == 0
        self.n_q, self.n_kv = n_q, n_kv
        self.head_dim = d_model // n_q
        self.rep = n_q // n_kv
        self.window, self.is_global, self.dropout = window, is_global, dropout
        q_dim, kv_dim = n_q * self.head_dim, n_kv * self.head_dim
        self.split = [q_dim, kv_dim, kv_dim]
        self.qkv_proj = nn.Linear(d_model, q_dim + 2 * kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, mask):
        B, T, C = x.shape
        Hq, Hkv, Dh, rep = self.n_q, self.n_kv, self.head_dim, self.rep
        q, k, v = torch.split(self.qkv_proj(x), self.split, dim=-1)
        q = q.view(B, T, Hq, Dh).transpose(1, 2)
        k = k.view(B, T, Hkv, Dh).transpose(1, 2)
        v = v.view(B, T, Hkv, Dh).transpose(1, 2)

        cos, sin = cos.to(q.dtype), sin.to(q.dtype)
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        dp = self.dropout if self.training else 0.0

        if self.is_global:
            k = k[:, :, None].expand(B, Hkv, rep, T, Dh).reshape(B, Hq, T, Dh)
            v = v[:, :, None].expand(B, Hkv, rep, T, Dh).reshape(B, Hq, T, Dh)
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dp, is_causal=True)
            return self.out_proj(y.transpose(1, 2).reshape(B, T, C))

        W = self.window
        N = T // W
        qc = q.view(B, Hq, N, W, Dh).transpose(1, 2).reshape(B * N, Hq, W, Dh)

        def chunk_kv(t):
            tc = t.view(B, Hkv, N, W, Dh)
            prev = F.pad(tc[:, :, :-1], (0, 0, 0, 0, 1, 0))
            t2 = torch.cat([prev, tc], dim=3)
            t2 = t2.transpose(1, 2)[:, :, :, None]
            return t2.expand(B, N, Hkv, rep, 2 * W, Dh).reshape(B * N, Hq, 2 * W, Dh)

        y = F.scaled_dot_product_attention(qc, chunk_kv(k), chunk_kv(v),
                                           attn_mask=mask, dropout_p=dp)
        y = y.view(B, N, Hq, W, Dh).permute(0, 1, 3, 2, 4).reshape(B, T, C)
        return self.out_proj(y)


class Arch4Block(nn.Module):
    def __init__(self, cfg, idx):
        super().__init__()
        self.is_global = (idx + 1) % cfg["GLOBAL_EVERY"] == 0
        self.norm1 = nn.LayerNorm(cfg["D_MODEL"])
        self.attn = GQAAttention(cfg["D_MODEL"], cfg["N_Q_HEADS"], cfg["N_KV_HEADS"],
                                 cfg["WINDOW"], self.is_global, cfg["DROPOUT"])
        self.norm2 = nn.LayerNorm(cfg["D_MODEL"])
        self.ffn = SwiGLU(cfg["D_MODEL"], cfg["D_FF"])

    def forward(self, x, cos, sin, mask):
        x = x + self.attn(self.norm1(x), cos, sin, mask)
        return x + self.ffn(self.norm2(x))


class Architecture4(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        W = cfg["WINDOW"]
        self.window = W
        assert cfg["MAX_SEQ_LEN"] % W == 0 and cfg["SEQ_LEN"] % W == 0

        self.token_embedding = nn.Embedding(cfg["VOCAB_SIZE"], cfg["D_MODEL"])
        self.layers = nn.ModuleList([Arch4Block(cfg, i) for i in range(cfg["N_LAYERS"])])
        self.final_norm = nn.LayerNorm(cfg["D_MODEL"])
        self.lm_head = nn.Linear(cfg["D_MODEL"], cfg["VOCAB_SIZE"], bias=False)
        self.lm_head.weight = self.token_embedding.weight

        self.rope = RotaryEmbedding(cfg["D_MODEL"] // cfg["N_Q_HEADS"],
                                    cfg["MAX_SEQ_LEN"], cfg["ROPE_BASE"])
        n_chunks = cfg["MAX_SEQ_LEN"] // W
        i = torch.arange(W)[:, None]
        j = torch.arange(2 * W)[None, :]
        m = (j > i) & (j <= i + W)
        mask = m[None].repeat(n_chunks, 1, 1)
        mask[0] &= (j >= W)
        self.register_buffer("chunk_mask", mask[:, None], persistent=False)

        self._init_weights()

    def _init_weights(self):
        std = self.cfg["INIT_STD"]
        nn.init.normal_(self.token_embedding.weight, 0.0, std)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.weight is not self.token_embedding.weight:
                nn.init.normal_(m.weight, 0.0, std)
        s = 1.0 / math.sqrt(2 * self.cfg["N_LAYERS"])
        for b in self.layers:
            b.attn.out_proj.weight.data.mul_(s)
            b.ffn.out_proj.weight.data.mul_(s)

    def hidden_states(self, input_ids):
        B, T = input_ids.shape
        W = self.window
        assert T % W == 0
        N = T // W
        x = self.token_embedding(input_ids)
        cos, sin = self.rope(T)
        mask = self.chunk_mask[:N][None].expand(B, N, 1, W, 2 * W).reshape(B * N, 1, W, 2 * W)
        for b in self.layers:
            x = b(x, cos, sin, mask)
        return self.final_norm(x)

    def forward(self, input_ids):
        return self.lm_head(self.hidden_states(input_ids))


# Fallback config used only when no cfg is available in the checkpoint metadata.
DEFAULT_CFG = {
    "VOCAB_SIZE": 40000, "D_MODEL": 1024, "N_LAYERS": 12, "N_Q_HEADS": 16, "N_KV_HEADS": 4,
    "D_FF": 3328, "MAX_SEQ_LEN": 1024, "SEQ_LEN": 1024, "WINDOW": 256, "GLOBAL_EVERY": 3,
    "ROPE_BASE": 10000.0, "DROPOUT": 0.0, "INIT_STD": 0.02,
}

# ============================================================
# 1. SETTINGS
# ============================================================
MODE = "test"                                                    # "test" or "val"
GPU_INDEX = int(os.environ.get("GPU_INDEX", 0))
RANGE = (int(os.environ.get("GEN_START", 0)), int(os.environ.get("GEN_END", -1)))
N_VAL_PER_SOURCE = 20                                            # val mode: pairs per source

CKPT = os.environ.get("CKPT") or "/kaggle/input/models/achiver07/arch4-1/transformers/default/1/arch4_hybrid_gqa_195m_resume/best_model.pt"
TOKD = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1"
RAW = "/kaggle/input/datasets/achiver07/data-project"            # only for MODE="val"
OUT = "/kaggle/working/submission_arch4_best"
V, BLOCK, W = 40000, 1024, 256
dev = f"cuda:{GPU_INDEX}"
os.makedirs(OUT, exist_ok=True)

# Prompt handling, identical to Arch 3: the model sees the LAST PROMPT_KEEP prompt tokens and may generate up to MAX_NEW tokens,
# so the sequence never exceeds BLOCK. 768 is a multiple of W; both numbers are YOUR choice, the assert only checks they fit.
PROMPT_KEEP = 768
MAX_NEW = 256
assert PROMPT_KEEP + MAX_NEW <= BLOCK

# Decoding settings. TEMP=0 -> greedy. TOP_K=0 -> off. TOP_P=1.0 -> off.
# Same values as submissions 1-3 (kept so the submissions differ only in the model). Change them only if you choose to.
GEN = dict(TEMP=0.90, TOP_K=50, TOP_P=1.0, SEED=0)
GEN_TAG = f"t{GEN['TEMP']}_k{GEN['TOP_K']}_p{GEN['TOP_P']}_s{GEN['SEED']}"

# Outputs with fewer than 128 words: "resample" (regenerate with other seeds, up to 20 tries) | "pad" | "error"
SHORT_POLICY = "resample"

assert torch.cuda.is_available(), "turn on the GPU"
assert torch.cuda.device_count() > GPU_INDEX, f"GPU_INDEX {GPU_INDEX} not available (visible GPUs: {torch.cuda.device_count()})"
assert os.path.exists(CKPT), f"checkpoint not found: {CKPT}  (set the CKPT environment variable or edit CKPT above)"
torch.cuda.set_device(GPU_INDEX)

# ============================================================
# 2. LOAD + START-UP CHECKS
# ============================================================
tok = Tokenizer.from_file(f"{TOKD}/tokenizer.json")
test = json.load(open(f"{TOKD}/test.json"))

ck = torch.load(CKPT, map_location="cpu", weights_only=False)

# The new training script saves best_model.pt as a PLAIN state_dict.
# The older best.pt/latest.pt format is a COMBINED checkpoint with a "model" key.
if isinstance(ck, dict) and "model" in ck:
    state_dict = ck["model"]
    step_info = ck.get("step", "unknown")
    best_val_info = ck.get("best_val", "unknown")
    cfg = ck.get("cfg") or DEFAULT_CFG
else:
    state_dict = ck
    step_info = "plain state_dict"

    # best_meta.json is saved beside best_model.pt by the training script.
    meta_path = os.path.join(os.path.dirname(CKPT), "best_meta.json")
    meta = {}
    if os.path.exists(meta_path):
        try:
            with open(meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)
        except Exception as e:
            print(f"Warning: could not read metadata file: {e}")

    cfg = meta.get("cfg") or DEFAULT_CFG
    step_info = meta.get("step", step_info)
    best_val_info = meta.get("best_val", "unknown")

assert cfg["WINDOW"] == W and cfg["VOCAB_SIZE"] == V and cfg["MAX_SEQ_LEN"] >= BLOCK,     "checkpoint config differs from this script's W / V / BLOCK"

# Hard guards for the actual trained Architecture 4.
assert cfg["D_MODEL"] == 1024
assert cfg["N_LAYERS"] == 12
assert cfg["N_Q_HEADS"] == 16
assert cfg["N_KV_HEADS"] == 4
assert cfg["D_FF"] == 3328
assert cfg["WINDOW"] == 256
assert cfg["GLOBAL_EVERY"] == 3

model = Architecture4(cfg).to(dev)
model.load_state_dict(state_dict, strict=True)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
assert n_params == 195_151_872, f"unexpected Architecture 4 parameter count: {n_params:,}"
assert cfg["N_Q_HEADS"] == 16 and cfg["N_KV_HEADS"] == 4, "expected trained Arch-4 GQA configuration 16Q/4KV"

print(
    f"loaded {CKPT} | step {step_info} | best val {best_val_info} | "
    f"parameters {n_params:,} | GPU {torch.cuda.get_device_name(dev)}"
)
print(
    f"checkpoint format: {'combined checkpoint' if isinstance(ck, dict) and 'model' in ck else 'plain state_dict'}"
)
print(
    f"generation: PROMPT_KEEP={PROMPT_KEEP}, MAX_NEW={MAX_NEW}, "
    f"TEMP={GEN['TEMP']}, TOP_K={GEN['TOP_K']}, TOP_P={GEN['TOP_P']}"
)
print(
    "attention pattern:",
    " ".join("G" if b.is_global else "L" for b in model.layers),
    "| KV heads", cfg["N_KV_HEADS"],
    "| d_ff", cfg["D_FF"]
)
del ck

gen = torch.Generator(device=dev).manual_seed(GEN["SEED"])


@torch.no_grad()
def last_logits(x):
    """Logits for the next token after x (B, T), any T <= BLOCK. Right-pads to a multiple of W (the model is causal, so the padding
    cannot change earlier positions) and applies the output head to the last REAL position only."""
    B, T = x.shape
    Tp = ((T + W - 1) // W) * W
    if Tp != T:
        x = F.pad(x, (0, Tp - T), value=0)
    h = model.hidden_states(x)
    return model.lm_head(h[:, T - 1:T, :])[:, 0, :]


def ref_attention(att, x, cos, sin, window):
    """Plain reference: full T x T attention with an EXPLICIT boolean mask (window=None -> ordinary causal)."""
    B, T, C = x.shape
    q, k, v = torch.split(att.qkv_proj(x), att.split, dim=-1)
    q = q.view(B, T, att.n_q, att.head_dim).transpose(1, 2)
    k = k.view(B, T, att.n_kv, att.head_dim).transpose(1, 2)
    v = v.view(B, T, att.n_kv, att.head_dim).transpose(1, 2)
    q = q * cos + rotate_half(q) * sin
    k = k * cos + rotate_half(k) * sin
    k = k.repeat_interleave(att.rep, dim=1)
    v = v.repeat_interleave(att.rep, dim=1)
    i = torch.arange(T, device=x.device)[:, None]
    j = torch.arange(T, device=x.device)[None, :]
    allowed = j <= i
    if window is not None:
        allowed = allowed & ((i - j) < window)
    y = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed)
    return att.out_proj(y.transpose(1, 2).reshape(B, T, C))


with torch.no_grad():                                           # start-up checks, full fp32, no autocast
    T0 = 1024
    N0 = T0 // W
    # 2a. the chunk mask, expanded to T x T, is exactly "j <= i and i - j < W"
    M = model.chunk_mask[:N0, 0].cpu()                          # (N0, W, 2W)
    full = torch.zeros(T0, T0, dtype=torch.bool)
    for c in range(N0):
        qi, kj = torch.nonzero(M[c], as_tuple=True)
        key = (c - 1) * W + kj
        assert (key >= 0).all(), "chunk 0 can see a key before position 0"
        full[c * W + qi, key] = True
    ii = torch.arange(T0)[:, None]
    jj = torch.arange(T0)[None, :]
    band = (jj <= ii) & ((ii - jj) < W)
    assert torch.equal(full, band), "chunk mask is not the 256-token causal band"
    print("check 2a passed: chunked local mask == explicit causal band of 256 tokens (itself + 255 before it)")

    # 2b. chunked local attention and global attention equal the plain reference, using the REAL trained weights
    torch.manual_seed(0)
    x0 = torch.randn(1, T0, cfg["D_MODEL"], device=dev)
    cos, sin = model.rope(T0)
    mask = model.chunk_mask[:N0][None].expand(1, N0, 1, W, 2 * W).reshape(N0, 1, W, 2 * W)
    for li in (0, 1, 2, 5, 11):
        att = model.layers[li].attn
        got = att(x0, cos, sin, mask)
        want = ref_attention(att, x0, cos, sin, None if att.is_global else W)
        d = (got - want).abs().max().item()
        assert torch.allclose(got, want, atol=1e-4, rtol=1e-3), f"layer {li} ({'global' if att.is_global else 'local'}): max diff {d:.2e}"
        print(f"check 2b passed: layer {li:2d} ({'global' if att.is_global else 'local '}) matches the explicit-mask reference (max diff {d:.1e})")

    # 2c. last_logits (padding + one-position head) equals the full forward at the same position, at many lengths
    x1 = torch.randint(3, V, (1, T0), device=dev)
    fl = model(x1).float()
    for t in (0, 100, 255, 256, 511, 700, 767, 768, 900, 1023):
        a = last_logits(x1[:, :t + 1]).float()
        d = (a - fl[:, t]).abs().max().item()
        assert torch.allclose(a, fl[:, t], atol=1e-3, rtol=1e-3), f"prefix length {t + 1}: last_logits differs from forward, max diff {d:.2e}"
    print("check 2c passed: last_logits (padded prefix) == forward logits at the same position, 10 lengths from 1 to 1024")
    del x0, x1, fl, M, full, band
torch.cuda.empty_cache()


# ============================================================
# 3. DECODING
# ============================================================
def sample(lg):
    if GEN["TEMP"] == 0:
        return lg.argmax(-1, keepdim=True)
    lg = lg / GEN["TEMP"]
    if GEN["TOP_K"]:
        v, _ = torch.topk(lg, min(GEN["TOP_K"], lg.size(-1)))
        lg[lg < v[:, [-1]]] = -float("inf")
    p = F.softmax(lg, -1)
    if GEN["TOP_P"] < 1.0:
        sp, si = torch.sort(p, descending=True)
        cum = sp.cumsum(-1)
        sp[(cum - sp) > GEN["TOP_P"]] = 0
        p = torch.zeros_like(p).scatter_(1, si, sp)
        p = p / p.sum(-1, keepdim=True)
    return torch.multinomial(p, 1, generator=gen)


@torch.no_grad()
def continue_text(prompt):
    full_ids = tok.encode(prompt).ids
    ids = full_ids[-PROMPT_KEEP:]
    x = torch.tensor([ids], device=dev)
    new = []
    while len(new) < MAX_NEW:
        with torch.autocast("cuda", dtype=torch.float16):
            lg = last_logits(x).float()
        nxt = sample(lg)
        x = torch.cat([x, nxt], 1)
        new.append(nxt.item())
        if len(new) % 8 == 0 and len(tok.decode(new).split()) >= 128:
            break
    return tok.decode(new).split(), len(full_ids)               # new tokens only: the prompt is never included; length is BEFORE cropping


def run_generation(prompts, tag):
    """prompts: {id: prompt_text}. Resumable via a part file keyed by tag + decoding settings."""
    part = f"{OUT}/pred_{tag}_{GEN_TAG}.json"
    meta_path = f"{OUT}/meta_{tag}_{GEN_TAG}.json"
    pred = json.load(open(part, encoding="utf-8")) if os.path.exists(part) else {}
    meta = json.load(open(meta_path, encoding="utf-8")) if os.path.exists(meta_path) else {"short_first_try": {}, "resampled_ok": [], "src_tokens": {}, "policy": SHORT_POLICY}
    t0, todo = time.time(), [k for k in prompts if k not in pred]
    for n, k in enumerate(todo):
        words, n_src = continue_text(prompts[k])
        meta["src_tokens"][k] = n_src
        if len(words) < 128:
            meta["short_first_try"][k] = len(words)
            if SHORT_POLICY == "error":
                raise RuntimeError(f"ID {k}: only {len(words)} words generated")
            if SHORT_POLICY == "resample":
                state = gen.get_state()
                for t in range(1, 21):
                    gen.manual_seed(1000 + t)
                    words, _ = continue_text(prompts[k])
                    if len(words) >= 128:
                        meta["resampled_ok"].append(k)
                        break
                gen.set_state(state)                            # later prompts are unaffected by the retries
                if len(words) < 128:
                    raise RuntimeError(f"ID {k}: still short after 20 resamples ({len(words)} words)")
            elif SHORT_POLICY == "pad":
                assert words, f"ID {k}: no words at all"
                words = (words * (128 // len(words) + 1))[:128]
        pred[k] = {"text": " ".join(words[:128])}
        if (n + 1) % 25 == 0 or n + 1 == len(todo) or n < 3:
            json.dump(pred, open(part, "w", encoding="utf-8"), ensure_ascii=False); json.dump(meta, open(meta_path, "w", encoding="utf-8"), ensure_ascii=False)
            print(f"{len(pred)}/{len(prompts)} done | {(time.time()-t0)/(n+1):.1f}s per prompt | short on first try: {len(meta['short_first_try'])}", flush=True)
    json.dump(pred, open(part, "w", encoding="utf-8"), ensure_ascii=False); json.dump(meta, open(meta_path, "w", encoding="utf-8"), ensure_ascii=False)
    assert all(len(v["text"].split()) == 128 for v in pred.values()), "not exactly 128 words"
    print(f"prompts with more than {PROMPT_KEEP} tokens (cropped to the last {PROMPT_KEEP}): "
          f"{sum(v > PROMPT_KEEP for v in meta['src_tokens'].values())} | short on first try: {len(meta['short_first_try'])} "
          f"(policy {SHORT_POLICY}, resampled ok {len(meta['resampled_ok'])})")
    return pred


def try_merge():
    """Union of every pred_test_*_<GEN_TAG>.json part in OUT; writes prediction.json + submission.zip only when all test IDs are present."""
    merged = {}
    for fn in sorted(glob.glob(f"{OUT}/pred_test_*_{GEN_TAG}.json")):
        merged.update(json.load(open(fn, encoding="utf-8")))
    if set(merged) != set(test):
        print(f"parts in {OUT} cover {len(merged)}/{len(test)} IDs: no zip yet (the process that finishes last writes it)")
        return
    assert all(len(v["text"].split()) == 128 for v in merged.values()), "not exactly 128 words"
    assert not any(test[k]["text"][:60] in v["text"] for k, v in merged.items()), "prompt leaked"
    with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
        json.dump(merged, f, ensure_ascii=False)
    with zipfile.ZipFile(f"{OUT}/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")
    print("zip root:", zipfile.ZipFile(f"{OUT}/submission.zip").namelist(), "| IDs:", len(merged))


# ============================================================
# 4. RUN
# ============================================================
if MODE == "test":
    ids_all = list(test.keys())
    a = max(0, RANGE[0])
    requested_end = len(ids_all) if RANGE[1] < 0 else RANGE[1]
    b = min(requested_end, len(ids_all))
    assert 0 <= a <= b <= len(ids_all), f"invalid generation range: {a}..{b}"
    prompts = {k: test[k]["text"] for k in ids_all[a:b]}
    print(f"generating prompts {a}..{b} of {len(ids_all)} on {dev}")
    pred = run_generation(prompts, f"test_{a}_{b}")
    assert not any(test[k]["text"][:60] in v["text"] for k, v in pred.items()), "prompt leaked"
    print(f"checked {len(pred)} predictions: 128 words each, no prompt text")
    try_merge()

    # If every GPU worker has finished, verify the final merged submission.
    zpath = f"{OUT}/submission.zip"
    if os.path.exists(zpath):
        with zipfile.ZipFile(zpath) as z:
            names = z.namelist()
        merged = json.load(open(f"{OUT}/prediction.json", encoding="utf-8"))
        assert names == ["prediction.json"], f"unexpected zip contents: {names}"
        assert len(merged) == len(test), f"expected {len(test)} IDs, found {len(merged)}"
        assert {len(v["text"].split()) for v in merged.values()} == {128}
        print(f"READY TO UPLOAD: {zpath}")

elif MODE == "val":
    rng, pairs = random.Random(1), []                           # fixed seed: same pairs for every decoding setting
    for s in ["acts_val", "sc_val", "tribunals_val"]:
        c = 0
        with open(f"{RAW}/{s}.jsonl", encoding="utf-8") as f:
            for line in f:
                t = json.loads(line)["text"]
                w = t.split()
                if len(w) < 640 or any(ord(ch) > 127 for ch in t):   # test prompts are pure ASCII
                    continue
                i = rng.randint(0, len(w) - 640)
                pairs.append((s, " ".join(w[i:i + 512]), " ".join(w[i + 512:i + 640])))
                c += 1
                if c >= N_VAL_PER_SOURCE:
                    break
    prompts = {str(i): p for i, (_, p, _) in enumerate(pairs)}
    pred = run_generation(prompts, f"val_{N_VAL_PER_SOURCE}")
    refs = {str(i): {"text": r} for i, (_, _, r) in enumerate(pairs)}
    json.dump(pred, open(f"{OUT}/val_pred_{GEN_TAG}.json", "w"))
    json.dump(refs, open(f"{OUT}/val_ref.json", "w"))
    print(f"wrote val_pred_{GEN_TAG}.json and val_ref.json ({len(pairs)} pairs)")
    print("Score BERTScore F1 / ROUGE-L / BLEU-4 with the COURSE evaluator for the official numbers.")
else:
    raise ValueError("MODE must be 'test' or 'val'")

saved this cell as /kaggle/working/gen4.py
1000 prompts over 2 GPU(s): GPU0: 0-500, GPU1: 500-1000
[  1.0 min] GPU0: 3/500 done | 3.8s per prompt | short on first try: 0 | GPU1: 3/500 done | 3.7s per prompt | short on first try: 0
[  2.0 min] GPU0: 25/500 done | 3.8s per prompt | short on first try: 0 | GPU1: 25/500 done | 3.9s per prompt | short on first try: 0
[  3.0 min] GPU0: 25/500 done | 3.8s per prompt | short on first try: 0 | GPU1: 25/500 done | 3.9s per prompt | short on first try: 0
[  4.0 min] GPU0: 50/500 done | 3.9s per prompt | short on first try: 0 | GPU1: 50/500 done | 4.0s per prompt | short on first try: 0
[  5.0 min] GPU0: 50/500 done | 3.9s per prompt | short on first try: 0 | GPU1: 50/500 done | 4.0s per prompt | short on first try: 0
[  6.0 min] GPU0: 75/500 done | 3.9s per prompt | short on first try: 0 | GPU1: 75/500 done | 4.0s per prompt | short on first try: 0
[  7.0 min] GPU0: 100/500 done | 3.9s per prompt | short on first try: 0 | GPU1: 100/500 done | 4.0

SystemExit: 0

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
